# Machine Translation with a Transformer

In this lab, we set up an English to French translation model:

1. use a pretrained **encoder-decoder** transformer, [MarianMT](https://huggingface.co/Helsinki-NLP/opus-mt-en-fr);
2. understand what happens during training (*teacher forcing*) and generation (beam search);
3. measure translation quality with the BLEU score;
4. fine-tune the model on a corpus of novel translations, with a PyTorch training loop.

*Remember to enable the GPU: Runtime > Change runtime type > GPU.*

In [ ]:
!pip install -q datasets sacrebleu sentencepiece

In [ ]:
import random

import sacrebleu
import torch
import tqdm.auto
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Computing on: {device}")
torch.manual_seed(0)
random.seed(0)

## Loading the model

The University of Helsinki's `opus-mt` models are encoder-decoder transformers trained on the [OPUS](https://opus.nlpl.eu/) parallel corpora, one model per language pair.

In [ ]:
model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

*Inspect the model (`print(model.config)`, `print(model)`):*

- *how many layers do the encoder and the decoder have?*
- *what are the model dimension and the number of attention heads?*
- *where is the cross-attention? What is it for?*
- *what is the vocabulary size? What kind of tokenization is used?*

In [ ]:
# Your code here

### Solution

In [ ]:
print(model.config)
print(model.model.decoder.layers[0])
print(sum(p.numel() for p in model.parameters()) / 1e6, "million parameters")
print(tokenizer.tokenize("The neighbour's cat is extraordinarily lazy."))

- The encoder and the decoder each have 6 layers (`encoder_layers`, `decoder_layers`), of dimension 512 (`d_model`) with 8 attention heads.
- Each decoder layer contains a **causal** self-attention (`self_attn`) then a cross-attention (`encoder_attn`): queries come from the decoder, keys and values from the encoder's output. This is how the decoder "reads" the source sentence for every generated word.
- The vocabulary has about 60,000 subwords, obtained with SentencePiece (`▁` marks the start of a word).

## Translating

The `generate` method implements the decoding loop. By default, this model uses **beam search**: instead of only keeping the most probable word at each step (greedy decoding), it keeps the `num_beams` most probable translation beginnings and picks the best complete translation at the end.

In [ ]:
def translate(sentences: list[str], num_beams: int = 4, batch_size: int = 32) -> list[str]:
  model.eval()
  translations = []
  for i in range(0, len(sentences), batch_size):
    batch = tokenizer(sentences[i:i + batch_size], return_tensors="pt",
                      padding=True, truncation=True, max_length=128).to(device)
    with torch.no_grad():
      generated = model.generate(**batch, num_beams=num_beams, max_length=256)
    translations += tokenizer.batch_decode(generated, skip_special_tokens=True)
  return translations


examples = ["I think, therefore I am.",
            "The weather is lovely today, let's go for a walk.",
            "Could you please send me the report before Friday?"]
for source, target in zip(examples, translate(examples)):
  print(f"{source}\n  → {target}")

*Compare the translations obtained with `num_beams=1` (greedy decoding) and `num_beams=5`, for instance on ambiguous or long sentences. Do you see differences?*

In [ ]:
# Your code here

### Solution

In [ ]:
tricky = ["He saw her duck.",
          "The old man the boats.",
          "Time flies like an arrow; fruit flies like a banana.",
          "Although the committee had initially rejected the proposal, "
          "it eventually approved a revised version after months of negotiation."]
for source, greedy, beam in zip(tricky, translate(tricky, num_beams=1),
                                translate(tricky, num_beams=5)):
  print(f"{source}\n  greedy → {greedy}\n  beam   → {beam}")

## The data: novel translations

We use [OPUS Books](https://huggingface.co/datasets/Helsinki-NLP/opus_books): sentences from public domain novels aligned with their French translation. The literary style (dialogues, old-fashioned turns of phrase) differs from the texts the model was mostly trained on.

We keep 1,000 pairs for validation, 1,000 for testing, and 20,000 for training (so that the lab stays fast).

In [ ]:
books = load_dataset("Helsinki-NLP/opus_books", "en-fr", split="train")
books = books.shuffle(seed=0)
test_pairs = books.select(range(1000))
val_pairs = books.select(range(1000, 2000))
train_pairs = books.select(range(2000, 22000))
for pair in test_pairs.select(range(3))["translation"]:
  print(pair)

## Measuring quality: the BLEU score

The [BLEU](https://en.wikipedia.org/wiki/BLEU) score compares the *n*-grams (sequences of 1 to 4 words) of a machine translation with those of a reference translation. It ranges from 0 to 100; above 30, translations are generally understandable, above 50 of good quality.

It is an imperfect measure (a correct translation phrased differently is penalized), but it makes it possible to compare models on the same corpus.

*Write a function `bleu(pairs) -> float` that translates the English sentences of `pairs` and computes the BLEU score with `sacrebleu.corpus_bleu(translations, [references])`. Compute the pretrained model's score on the test set.*

In [ ]:
# Your code here

### Solution

In [ ]:
def bleu(pairs, num_beams: int = 4) -> float:
  sources = [pair["en"] for pair in pairs["translation"]]
  references = [pair["fr"] for pair in pairs["translation"]]
  return sacrebleu.corpus_bleu(translate(sources, num_beams=num_beams), [references]).score


bleu_before = bleu(test_pairs)
print(f"BLEU before fine-tuning: {bleu_before:.1f}")

## Under the hood: *teacher forcing*

During training, we don't generate word by word: the decoder is given the **reference translation** shifted by one position, and asked to predict each next word. Thanks to the causal mask, all positions are predicted in a single pass.

The tokenizer prepares both sides: `text_target` produces the `labels`, and the model computes the cross-entropy loss itself when given these `labels`.

*Compute the model's loss on a sentence pair. Print the shape of the `logits`: what does each dimension correspond to?*

In [ ]:
pair = train_pairs[0]["translation"]
batch = tokenizer(pair["en"], text_target=pair["fr"], return_tensors="pt").to(device)
print(batch.keys())
# Your code here

### Solution

In [ ]:
with torch.no_grad():
  output = model(**batch)
print(f"Loss: {output.loss.item():.3f}")
print(f"Logits shape: {tuple(output.logits.shape)}")
print(tokenizer.convert_ids_to_tokens(batch["labels"][0]))

The logits have shape `(batch, translation length, vocabulary size)`: for each position of the reference translation, one score per vocabulary subword.

## Preparing the training data

*Write a function `preprocess(examples)` that tokenizes a batch of dataset examples (English sources and French targets, truncated to 128 tokens), then apply it with `train_pairs.map(preprocess, batched=True, remove_columns=...)`.*

*The provided `DataCollatorForSeq2Seq` then pads the sequences of a batch. Why does it replace the padding of the `labels` with `-100`?*

In [ ]:
# Your code here

### Solution

In [ ]:
def preprocess(examples):
  sources = [pair["en"] for pair in examples["translation"]]
  targets = [pair["fr"] for pair in examples["translation"]]
  return tokenizer(sources, text_target=targets, max_length=128, truncation=True)


train_tokenized = train_pairs.map(preprocess, batched=True,
                                  remove_columns=train_pairs.column_names)
print(train_tokenized)

`-100` is the index `cross_entropy` ignores (`ignore_index`): padding positions don't count in the loss.

In [ ]:
collator = DataCollatorForSeq2Seq(tokenizer, model=model)
train_loader = DataLoader(train_tokenized, batch_size=32, shuffle=True, collate_fn=collator)
batch = next(iter(train_loader))
print({key: value.shape for key, value in batch.items()})

## Fine-tuning

*Write the training loop for one epoch:*

- *`AdamW` optimizer, learning rate of `5e-5` (small: we are adjusting an already good model);*
- *a linear learning rate schedule with *warmup* (`transformers.get_linear_schedule_with_warmup`, 10% of the steps as warmup);*
- *the loss is `model(**batch).loss`;*
- *print the average loss every 100 steps.*

*On a Colab GPU, one epoch takes a few minutes.*

In [ ]:
# Your code here

### Solution

In [ ]:
from transformers import get_linear_schedule_with_warmup

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
n_steps = len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=n_steps // 10,
                                            num_training_steps=n_steps)

model.train()
running_loss = 0.0
for step, batch in enumerate(tqdm.auto.tqdm(train_loader), 1):
  batch = {key: value.to(device) for key, value in batch.items()}
  optimizer.zero_grad()
  loss = model(**batch).loss
  loss.backward()
  optimizer.step()
  scheduler.step()
  running_loss += loss.item()
  if step % 100 == 0:
    print(f"Step {step}: loss {running_loss / 100:.3f}")
    running_loss = 0.0

## Evaluation after fine-tuning

*Compute the fine-tuned model's BLEU score on the test set and compare it with the score before fine-tuning. Also compare a few translations before/after. What do you notice?*

In [ ]:
# Your code here

### Solution

In [ ]:
bleu_after = bleu(test_pairs)
print(f"BLEU before: {bleu_before:.1f}, after: {bleu_after:.1f}")

for pair in test_pairs.select(range(5))["translation"]:
  print(f"Source    : {pair['en']}")
  print(f"Reference : {pair['fr']}")
  print(f"Model     : {translate([pair['en']])[0]}")
  print("-" * 80)

The BLEU score improves after a single epoch (in a reduced trial, with only 640 training pairs, it went from 22.4 to 23.5): the model adapts to the style and vocabulary of novels. The fine-tuned translations also follow the reference's phrasing more closely. This is transfer learning applied to text: start from a general model and adjust it with a little data from the target domain.

To go further:

- check on the validation set that the model doesn't overfit if you train longer;
- save the fine-tuned model with `model.save_pretrained("opus-mt-en-fr-books")` and `tokenizer.save_pretrained(...)`, then reload it with `from_pretrained`.